In [0]:
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# 1. Schema
telemetry_schema = StructType(
    [
        StructField("timestamp", TimestampType(), True),
        StructField("cpu_utilization", DoubleType(), True),
        StructField("gpu_utilization", DoubleType(), True),
        StructField("power_draw_watts", DoubleType(), True),
        StructField("current_fluid_temp", DoubleType(), True),
    ]
)

# 2. Paths (Replace <your_volume_name> with your exact Volume name)
source_path = "/Volumes/workspace/default/thermal_data/"
checkpoint_path = "/Volumes/workspace/default/thermal_data/_checkpoints/bronze/"
bronze_table_name = "bronze_telemetry"

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
from pyspark.sql import functions as F

# 1. Read stream using Auto Loader
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(telemetry_schema)
    .load(source_path)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

# 2. Write stream to Delta Bronze table
query = (
    bronze_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table_name)
)

query.awaitTermination()

In [0]:
display(spark.table(bronze_table_name))